# OSM: educación

Análisis exploratorio independiente de bronce. Fecha de revisión: 13 de noviembre de 2026.

Se examinan los archivos locales originales, no una tabla Delta ejecutada en Databricks.
No se limpia ni sobrescribe la fuente. Las transformaciones que aparezcan se hacen en copias
para evaluar lo que necesitará plata. Los resultados no prueban causalidad ni certifican seguridad.

Cada notebook sigue el orden: procedencia y unidad, calidad por columna y geometría,
distribuciones, límites y reglas para plata. No se muestran registros individuales de edificios oficiales.

In [ ]:
from pathlib import Path
import sys, json, re, unicodedata
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from IPython.display import display, HTML

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "CLAUDE.md").exists() and (p / "data/externos").is_dir())
sys.path.insert(0, str(ROOT / "analisis_bronce"))
from soporte import (ficha, huella, leer_geojson, mostrar_perfil, geometria, conteos,
                     faltantes, guardar_resumen, grafico_barras, SALIDAS, CRS_METROS)
pd.set_option("display.max_columns", 24)
pd.set_option("display.max_rows", 60)
plt.rcParams.update({"font.size": 11, "figure.dpi": 110})

## 1. Procedencia y unidad de análisis

Archivo: `data/externos/osm_educacion.geojson`. Una fila representa un sitio de educación registrado en OSM.
La fecha de descarga no equivale a fecha de verificación del objeto en terreno.

In [ ]:
PATH = ROOT / 'data/externos/osm_educacion.geojson'
meta = ficha(PATH)
raw, df, g = leer_geojson(PATH)
print("Archivo:", meta["archivo"], "| filas:", len(df), "| propiedades:", len(df.columns))
display(pd.Series(meta, name="procedencia").to_frame())

## 2. Calidad por columna y geometría

In [ ]:
calidad = mostrar_perfil(df)
geo = geometria(g)
display(pd.Series(geo, name="geometria").to_frame())
resumen = {"dataset": '06_osm_educacion', **meta, "filas": len(df), "columnas": list(df.columns),
           "calidad_columnas": calidad.to_dict("records"), "geometria": geo}

## 3. Categorías y evidencia sobre usos de emergencia

In [ ]:
tipos=conteos(df.amenity,"tipo_educacion")
display(tipos)
display(faltantes(df,["name","amenity","emergency","shelter_type","capacity","operator","check_date"]))
grafico_barras(tipos,"Sitios de educación disponibles", "06_tipos")
resumen.update({"tipos":df.amenity.value_counts().to_dict(),"osm_id_duplicados":int(df.osm_id.duplicated().sum()),
                "atributos_albergue_presentes":[c for c in ["emergency","shelter_type","capacity"] if c in df]})

## Reglas decididas para plata

- **Conservar como equipamientos educativos de contexto**, sin incorporarlos como albergues habilitados al cálculo de recursos.
- Crear tipología `school/university/college`; conservar nombre e identificador de fuente.
- La selección como albergue requiere autorización, condición estructural, capacidad, servicios y accesibilidad que esta capa no acredita.
- Su uso como capa de exposición educativa puede ampliarse después; no mejora por sí sola el inventario de capacidad de respuesta.

In [ ]:
assert huella(PATH) == meta["sha256"], "La fuente cruda cambió durante el análisis"
guardar_resumen('06_osm_educacion', resumen)
print("Resumen y gráficos guardados; la fuente cruda conserva su SHA-256.")